# Código atualizado


In [5]:
# ============================================
# 0) Imports
# ============================================
import warnings

import numpy as np
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier 

import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)


In [6]:
RNG_SEED = 42

OUTPUT_DIR = 'output'

DATASET_K8_TSALLIS = OUTPUT_DIR + '/dataset_k8_tsallis.csv'
DATASET_K11_TSALLIS = OUTPUT_DIR + '/dataset_k11_tsallis.csv'
DATASET_K16_TSALLIS = OUTPUT_DIR + '/dataset_k16_tsallis.csv'

RESULT_K8_TSALLIS = OUTPUT_DIR + '/resultados_k8_tsallis.csv'
RESULT_K11_TSALLIS = OUTPUT_DIR + '/resultados_k11_tsallis.csv'
RESULT_K16_TSALLIS = OUTPUT_DIR + '/resultados_k16_tsallis.csv'

np.random.seed(RNG_SEED)


In [7]:
# ============================================
# 1) Carregar datasets
# ============================================
df_k8_tsallis = pd.read_csv(DATASET_K8_TSALLIS)
df_k11_tsallis = pd.read_csv(DATASET_K11_TSALLIS)
df_k16_tsallis = pd.read_csv(DATASET_K16_TSALLIS)

print("Shape k8_tsallis: ", df_k8_tsallis.shape)
print("Shape k11_tsallis: ", df_k11_tsallis.shape)
print("Shape k16_tsallis: ", df_k16_tsallis.shape)

Shape k8_tsallis:  (81, 12)
Shape k11_tsallis:  (81, 15)
Shape k16_tsallis:  (81, 20)


In [8]:
# ============================================
# 2) Preparar X, y e grupos
# Removemos metadados e o target das features
# ============================================

cols_to_drop = ["status", "subject_id"]
feature_cols_k8_tsallis = [c for c in df_k8_tsallis.columns if c not in cols_to_drop]
feature_cols_k11_tsallis = [c for c in df_k11_tsallis.columns if c not in cols_to_drop]
feature_cols_k16_tsallis = [c for c in df_k16_tsallis.columns if c not in cols_to_drop]

X_k8_tsallis = df_k8_tsallis[feature_cols_k8_tsallis]
X_k11_tsallis = df_k11_tsallis[feature_cols_k11_tsallis]
X_k16_tsallis = df_k16_tsallis[feature_cols_k16_tsallis]

y = df_k16_tsallis["status"].astype(int)
groups = df_k16_tsallis["subject_id"]

print("Colunas a serem descartadas:", cols_to_drop)
print("Colunas de features k8_tsallis:", feature_cols_k8_tsallis)
print(len(feature_cols_k8_tsallis), "features selecionadas.")
print("Colunas de features k11_tsallis:", feature_cols_k11_tsallis)
print(len(feature_cols_k11_tsallis), "features selecionadas.")
print("Colunas de features k16_tsallis:", feature_cols_k16_tsallis)
print(len(feature_cols_k16_tsallis), "features selecionadas.")

print("\nBalanceamento:")
print(pd.Series(y).value_counts().rename(index={0: "Controle(0)", 1: "Parkinson(1)"}))

Colunas a serem descartadas: ['status', 'subject_id']
Colunas de features k8_tsallis: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std', 'tsallis_sq_amp', 'tsallis_sq_f0']
10 features selecionadas.
Colunas de features k11_tsallis: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std', 'mfcc8_std', 'dmfcc3_mean', 'mfcc3_std', 'tsallis_sq_amp', 'tsallis_sq_f0']
13 features selecionadas.
Colunas de features k16_tsallis: ['dmfcc6_std', 'mfcc13_std', 'mfcc10_std', 'mfcc11_std', 'dmfcc11_std', 'mfcc2_mean', 'spec_centroid_mean_hz', 'mfcc9_std', 'mfcc8_std', 'dmfcc3_mean', 'mfcc3_std', 'shimmer_apq3', 'dmfcc13_mean', 'dmfcc4_std', 'jitter_ppq5', 'mfcc1_std', 'tsallis_sq_amp', 'tsallis_sq_f0']
18 features selecionadas.

Balanceamento:
status
Controle(0)     41
Parkinson(1)    40
Name: count, dtype: int64


In [9]:
def make_pipeline(model):
    return ImbPipeline(steps=[  
        # 1. RobustScaler: Escalonamento robusto baseado em quartis
        ("scale", RobustScaler()),

        # 2. SMOTE: Balanceamento sintético aplicado apenas durante o 'fit' (treino)
        ("smote", SMOTE(random_state=RNG_SEED, k_neighbors=3)),

        # 3. O Classificador (SVC, XGBoost, etc.)
        ("clf", model)
    ])
    

In [10]:
# =========================================================
# 13) Benchmark: roda GroupKFold CV e devolve tabela resumo
# =========================================================
def benchmark_models(X, y, groups, models_dict, n_splits=5, n_jobs=1):
    cv = GroupKFold(n_splits=n_splits)

    scoring = {
        "bal_acc": "balanced_accuracy",
        "roc_auc": "roc_auc",
        "f1": "f1",
    }

    rows = []
    for name, model in models_dict.items():
        pipe = make_pipeline(model)

        scores = cross_validate(
            pipe, X, y,
            groups=groups,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs,
            error_score="raise"
        )

        rows.append({
            "model": name,
            "bal_acc_mean": float(np.mean(scores["test_bal_acc"])),
            "bal_acc_std":  float(np.std(scores["test_bal_acc"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std":  float(np.std(scores["test_roc_auc"])),
            "f1_mean":      float(np.mean(scores["test_f1"])),
            "f1_std":       float(np.std(scores["test_f1"])),
        })

    return pd.DataFrame(rows).sort_values("bal_acc_mean", ascending=False).reset_index(drop=True)


In [11]:
# =========================================================
# 15) Modelos Classicos - Ajuste fino
# =========================================================


models = {
    #0.713889 
    "logreg_l2": LogisticRegression(
            C=0.5,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),
    # 0.640278
    "svc_rbf":  SVC(
            C=1.0,
            gamma="scale",
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),
    # 0.663889
    "linear_svc_cal": CalibratedClassifierCV(
            estimator=LinearSVC(
                C=0.1,
                class_weight="balanced",
                random_state=RNG_SEED,
                max_iter=10000,
                dual=False
            ),
            method="sigmoid",
            cv=3,
        ),
    # 0.726389
    "random_forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_split=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RNG_SEED,
            n_jobs=-1,
        ),
    # 0.640278
    "gradient_boosting": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.02,
            max_depth=2,
            subsample=0.6,          # Treina com apenas 60% das linhas por árvore (estocástico)
            random_state=RNG_SEED,
        ),
    # 0.727778 -> adaboost      0.702778     0.046356      0.848785     0.054560
    "adaboost": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RNG_SEED),
            n_estimators=100,
            learning_rate=0.5,            
            random_state=RNG_SEED,
        ),
    # 0.541667
    "decision_tree": DecisionTreeClassifier(
            max_depth=2,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RNG_SEED,
        ),
    # -------------------------
    # KNN (não tem random_state)
    # -------------------------
    # 0.740278
    "knn": KNeighborsClassifier(
            n_neighbors=7,
            weights="uniform",
            metric="minkowski",
            p=2,
        ),
    
    # -------------------------
    # MLP (rede rasa)
    # -------------------------
    # 0.640278
    "mlp": MLPClassifier(
            hidden_layer_sizes=(4,),
            solver='lbfgs',
            alpha=1.0,
            max_iter=500,
            random_state=RNG_SEED,
            early_stopping=True,
            validation_fraction=0.1
        ),
    # -------------------------
    # XGBoost / CatBoost
    # -------------------------
    # 0.738889 -> xgboost      0.738889     0.047222      0.788021     0.028603
    "xgboost": XGBClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.02,
            subsample=0.6,          # Cada árvore vê apenas 60% das 81 linhas
            colsample_bytree=0.6,   # Cada árvore vê apenas 60% das 16 colunas
            # 3. Regularização Pesada (Os "Freios")
            gamma=2,                # Exige uma redução mínima de perda para criar um novo corte (poda)
            reg_alpha=1.0,          # Regularização L1 (torna os pesos esparsos)
            reg_lambda=2.0,         # Regularização L2 (achata os pesos)
            min_child_weight=5,
            objective="binary:logistic",
            random_state=RNG_SEED,
            n_jobs=-1,
            verbosity=0,
        ),
    # 0.676389 ->   0.726389||0.116253  0.851042||0.067153
    "catboost": CatBoostClassifier(
        iterations=200, 
        depth=2, 
        learning_rate=0.02,
        l2_leaf_reg=10,          # Aumenta regularização L2
        bootstrap_type='Bernoulli',
        subsample=0.5,
        loss_function='Logloss', 
        verbose=0, 
        random_seed=RNG_SEED,
        allow_writing_files=False # Evita criar a pasta catboost_info
    ),

    # 0.663889
    "lightgbm": LGBMClassifier(
        n_estimators=100, 
        learning_rate=0.02, 
        num_leaves=3,
        min_child_samples=5,
        boosting_type='goss',
        random_state=RNG_SEED, 
        n_jobs=-1, 
        colsample_bytree=0.5,
        importance_type='gain',
        verbosity=-1,

    )

}


In [12]:
# =========================================================
# 16) Rodar benchmark (GroupKFold por sujeito)
# =========================================================
results_k8_tsallis = benchmark_models(X_k8_tsallis, y, groups, models, n_splits=5, n_jobs=1)
results_k11_tsallis = benchmark_models(X_k11_tsallis, y, groups, models, n_splits=5, n_jobs=1)
results_k16_tsallis = benchmark_models(X_k16_tsallis, y, groups, models, n_splits=5, n_jobs=1)

# Mostrar ranking
print("\n=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===")
print("\n=== Ranking k8_tsallis ===")
print(results_k8_tsallis)
print("\n=== Ranking k11_tsallis ===")
print(results_k11_tsallis)
print("\n=== Ranking k16_tsallis ===")
print(results_k16_tsallis)


=== Ranking por Balanced Accuracy (GroupKFold, sem leakage) ===

=== Ranking k8_tsallis ===
                model  bal_acc_mean  bal_acc_std  roc_auc_mean  roc_auc_std  \
0            catboost      0.777778     0.092962      0.783333     0.056921   
1             xgboost      0.765278     0.046356      0.807118     0.025017   
2   gradient_boosting      0.754167     0.082706      0.755903     0.096591   
3             svc_rbf      0.752778     0.118715      0.719097     0.102753   
4            lightgbm      0.740278     0.090502      0.764931     0.045067   
5                 knn      0.740278     0.044052      0.772569     0.077727   
6       random_forest      0.740278     0.100500      0.794792     0.057773   
7      linear_svc_cal      0.738889     0.100212      0.786806     0.101603   
8       decision_tree      0.704167     0.051707      0.725000     0.065996   
9           logreg_l2      0.688889     0.125031      0.771875     0.102508   
10                mlp      0.680556   

In [13]:
# -------------- salvando resultados ------------------
results_k8_tsallis.to_csv(RESULT_K8_TSALLIS, index=False)
print(f'Resultados salvos no arquivo {RESULT_K8_TSALLIS}')

results_k11_tsallis.to_csv(RESULT_K11_TSALLIS, index=False)
print(f'Resultados salvos no arquivo {RESULT_K11_TSALLIS}')

results_k16_tsallis.to_csv(RESULT_K16_TSALLIS, index=False)
print(f'Resultados salvos no arquivo {RESULT_K16_TSALLIS}')

Resultados salvos no arquivo output/resultados_k8_tsallis.csv
Resultados salvos no arquivo output/resultados_k11_tsallis.csv
Resultados salvos no arquivo output/resultados_k16_tsallis.csv
